In [1]:
import cadquery as cq
from cadquery import exporters
import math
from jupyter_cadquery import *

from jupyter_cadquery.replay import replay, enable_replay, disable_replay

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

versions()

Overwriting auto display for cadquery Workplane and Shape

Versions:
- jupyter_cadquery  4.0.2
- cad_viewer_widget 3.0.2
- open cascade      7.7.2.1



In [2]:
cv = open_viewer("Girder", cad_width=800, height=600)
set_default_viewer("Girder")

In [3]:
# ------------------------------------------------------------
# PLATE -> GIRDER DOVETAIL NOTCH
#
# World coordinates:
#   X = along the plate
#   Y = along the girder
#   Z = vertical
#
# Plate runs along X.
# Girder approaches perpendicular to the plate along Y.
# Girder and plate tops are flush.
# ------------------------------------------------------------


# ------------------------------------------------------------
# PLATE
# ------------------------------------------------------------

plate_length = 72.0       # X
plate_width = 4.0         # Y
plate_height = 6.0        # Z


# ------------------------------------------------------------
# GIRDER
# ------------------------------------------------------------

girder_length = 48.0      # Y
girder_width = 6.0        # X
girder_height = 6.0       # Z


# ------------------------------------------------------------
# DOVETAIL
# ------------------------------------------------------------

# Vertical depth of the dovetail into the top of the plate
dovetail_depth = 3.0

# Narrow width of the dovetail at the neck
dovetail_neck_width = 3.0     # X

# Wider width at the bottom of the dovetail
dovetail_bottom_width = 4.0   # X

# How far the girder end enters the plate
dovetail_length = plate_width

# Small machining / Boolean clearance
dovetail_clearance = 0.02


# ------------------------------------------------------------
# PLACEMENT
# ------------------------------------------------------------

# Put the joint at the center of the plate for the prototype
joint_x = 0.0

# Plate centered on Y=0
plate_y0 = -plate_width / 2.0

# Top of plate establishes our common elevation
plate_top_z = plate_height

In [4]:
# ------------------------------------------------------------
# RAW PLATE AND GIRDER BLANKS
# ------------------------------------------------------------

# Plate centered on X=0 and Y=0
plate = cq.Solid.makeBox(
    plate_length,
    plate_width,
    plate_height,
    cq.Vector(
        -plate_length / 2.0,
        -plate_width / 2.0,
        0.0,
    ),
)


# Girder centered on X=joint_x.
# It approaches the plate from negative Y and terminates
# at the far side of the plate.
girder_x0 = joint_x - girder_width / 2.0
girder_y0 = -girder_length + plate_width / 2.0
girder_z0 = plate_top_z - girder_height

girder = cq.Solid.makeBox(
    girder_width,
    girder_length,
    girder_height,
    cq.Vector(
        girder_x0,
        girder_y0,
        girder_z0,
    ),
)

In [5]:
# ------------------------------------------------------------
# GIRDER DOVETAIL
# ------------------------------------------------------------

# Plate faces along Y
girder_shoulder_y = -plate_width / 2.0
dovetail_end_y = girder_shoulder_y + dovetail_length

# Bottom of the dovetail tail
dovetail_z0 = plate_top_z - dovetail_depth


# ------------------------------------------------------------
# REMOVE THE RAW GIRDER END INSIDE THE PLATE
# ------------------------------------------------------------

cut_margin = 1.0

girder_end_cutter = cq.Solid.makeBox(
    girder_width + 2.0 * cut_margin,
    dovetail_length + cut_margin,
    girder_height + 2.0 * cut_margin,
    cq.Vector(
        girder_x0 - cut_margin,
        girder_shoulder_y,
        girder_z0 - cut_margin,
    ),
)

girder_body = (
    girder
    .cut(girder_end_cutter)
    .clean()
)


# ------------------------------------------------------------
# DOVETAIL TAIL
#
# Narrow at the shoulder, wider toward the inserted end.
# ------------------------------------------------------------

dovetail_profile = (
    cq.Workplane("XY")
    .moveTo(
        joint_x - dovetail_neck_width / 2.0,
        girder_shoulder_y,
    )
    .lineTo(
        joint_x + dovetail_neck_width / 2.0,
        girder_shoulder_y,
    )
    .lineTo(
        joint_x + dovetail_bottom_width / 2.0,
        dovetail_end_y,
    )
    .lineTo(
        joint_x - dovetail_bottom_width / 2.0,
        dovetail_end_y,
    )
    .close()
)

dovetail_tail = (
    dovetail_profile
    .extrude(dovetail_depth)
    .translate((0, 0, dovetail_z0))
    .val()
)


# ------------------------------------------------------------
# FINISHED GIRDER
# ------------------------------------------------------------

girder_dovetail = (
    girder_body
    .fuse(dovetail_tail)
    .clean()
)

In [6]:
# ------------------------------------------------------------
# MATCHING DOVETAIL MORTISE IN PLATE
# ------------------------------------------------------------

mortise_overlap = 0.05

mortise_neck_width = (
    dovetail_neck_width
    + dovetail_clearance
)

mortise_bottom_width = (
    dovetail_bottom_width
    + dovetail_clearance
)

mortise_y0 = (
    girder_shoulder_y
    - mortise_overlap
)

mortise_y1 = (
    dovetail_end_y
    + mortise_overlap
)


# ------------------------------------------------------------
# DOVETAIL MORTISE PROFILE
# ------------------------------------------------------------

mortise_profile = (
    cq.Workplane("XY")
    .moveTo(
        joint_x - mortise_neck_width / 2.0,
        mortise_y0,
    )
    .lineTo(
        joint_x + mortise_neck_width / 2.0,
        mortise_y0,
    )
    .lineTo(
        joint_x + mortise_bottom_width / 2.0,
        mortise_y1,
    )
    .lineTo(
        joint_x - mortise_bottom_width / 2.0,
        mortise_y1,
    )
    .close()
)


# ------------------------------------------------------------
# MORTISE CUTTER
#
# Extend slightly below the tail and above the plate top
# to guarantee a clean Boolean opening.
# ------------------------------------------------------------

mortise_cutter = (
    mortise_profile
    .extrude(
        dovetail_depth
        + 2.0 * mortise_overlap
    )
    .translate((
        0,
        0,
        dovetail_z0 - mortise_overlap,
    ))
    .val()
)


# ------------------------------------------------------------
# CUT THE PLATE
# ------------------------------------------------------------

plate_dovetail = (
    plate
    .cut(mortise_cutter)
    .clean()
)

In [7]:
show_object(plate_dovetail, name="plate")
show_object(girder_dovetail, name="girder")

+
